In [56]:
#pip install langchain langchain-community langchain-core

In [58]:
#pip install pypdf

In [60]:
#pip install langchain-text-splitters

In [68]:
#pip install langchain-huggingface sentence-transformers

In [62]:
#pip install langchain-ollama

In [64]:
#pip install -U langchain-huggingface sentence-transformers

In [66]:
#pip install faiss-cpu

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import faiss

C:\Users\avina\AppData\Local\Temp\ipykernel_41936\605700651.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# Load the document 

In [7]:
file_path='project_paper.pdf'
loader=PyPDFLoader(file_path)
pages=loader.load()
print(f'loaded {len(pages)} pages')


loaded 14 pages


In [ ]:
#chunking


In [9]:
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=100)
split_doc=splitter.split_documents(pages)
print(f'length of the split documents is {len(split_doc)}')


length of the split documents is 61


In [ ]:
# create the embedding and faiss-index


In [15]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [21]:
dimension=384
index=faiss.IndexFlatL2(dimension)

vectore_store=FAISS(embedding_function=embeddings,index=index,docstore=InMemoryDocstore(),index_to_docstore_id={})
vectore_store.add_documents(split_doc)
print('documents embedded and indexed')

documents embedded and indexed


In [27]:
retriver=vectore_store.as_retriever(search_kwargs={'k':4})
retriver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023A8821F560>, search_kwargs={'k': 4})

In [ ]:
#create LLM


In [50]:
llm=ChatOllama(model='qwen3:8b ',temperature=0.8)
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14'}}, output_version=None, model='qwen3:8b ', temperature=0.8)

In [37]:
#Build a custom retrival +prompt chain
query='explain the what document is about?'

In [52]:
#Retrive documents
retrive_doc=retriver.invoke(query)


In [41]:
context_text='\n\n'.join([doc.page_content for doc in retrive_doc])
context_text

'usage do not progressively increase during the generation process.\n3. Methodology\n3.1. Long-horizon Parsing\nOur humans excel at long-horizon parsing tasks—continuously transcribing an entire book,\ntranslating even hundreds of pages in one sitting, or transcribing hours of audio without\ninterruption. This continuous parsing capability appears closely linked to the working memory.\nAs illustrated in Figure 2, when a person copies a book by hand, their attention typically centers\non three points: the original source book, a small portion of what has just been written (usually\nonly a few characters), and the next character about to be written. Rather than retaining a\ncomplete memory of everything already transcribed, they engage in a form of soft forgetting.\n\n1. Introduction\nHumans are remarkably adept at seemingly straightforward long-horizon tasks: transcribing\nhundreds of book pages, translating hours-long audio recordings, and the like. Yet these are\nprecisely the tasks w

In [44]:
#create prompt template
prompt=ChatPromptTemplate.from_template("""
You are an expert research assistant.

Answer the question only from the provided context.

Context:
{context}

Question:
{question}

Answer clear and concisely.
""")

In [46]:
formatted_prompt=prompt.format(context=context_text,question=query)

In [54]:
response=llm.invoke(formatted_prompt)
print('\n Answer:\n',response.content)


 Answer:
 The document discusses challenges in current AI models' ability to handle long-horizon tasks (e.g., transcribing books, translating audio) compared to human capabilities. It highlights that humans use **soft forgetting** and focus on **local context** (recently written text and upcoming characters) while maintaining a continuous cognitive state, unlike models that reset memory page-by-page. The paper critiques the for-loop paradigm as an inefficient workaround and proposes methodologies inspired by human parsing mechanisms (e.g., attention to nearby context, fading distant outputs) to enable more efficient, continuous processing for tasks like OCR and long-document handling, aiming toward AGI-like intelligence.
